# build a neural network from scratch: lenet-5 vs memes

every ai you've heard of, from chatgpt to self driving cars, is built out of neural networks. most people only ever touch them thru libraries like pytorch. in this notebook you build one yourself with **no ai libraries**, just python and math. you write every piece of lenet-5, train it on 35,000 memes, and then test it on 150 real memes it has never seen (yes that includes people doing the 6-7 hands).

### how this notebook works
- go top to bottom. run a cell with **shift + enter**.
- cells that start with `# todo:` are yours. the comments inside walk thru the logic step by step and list every bit of python you'll need. delete the `raise NotImplementedError(...)` line and write your code there, then run the check cell right under it.
- a check prints `pass`, `fail` (with what went wrong and a hint) or `todo` if you havent written it yet.
- stuck for more than a few minutes? put `%load solutions/NAME.py` in the cell and run it twice. the first run pastes the answer in and the second one runs it. if your editor doesnt paste it, use `%run -i solutions/NAME.py` instead.
- cells that start with `# given code` are already written. read the comments on top, then run them.
- you can only use the python standard library (`math`, `random`, `operator`, `time`, `pathlib`, ...). `viz` only draws pictures and `checks` only grades your answers, neither of them is part of the network.
- the next two cells are a **formula sheet** and a **python cheat sheet**. you dont have to read them now, come back whenever a todo mentions something you dont know.

| | what you build | time |
|---|---|---|
| lab 0 - the big picture | the dataset, lenet-5's shapes, how many weights it has | 10 min |
| lab 1 - layers by hand | convolution, relu, max-pool, fully connected, softmax with plain loops | 20 min |
| lab 2 - how data is stored | layouts (shape + stride), views, the im2col trick | 25 min |
| lab 3 - layers from tensors | a ~10x faster forward pass: one gather + one matrix multiply per layer | 15 min |
| lab 4 - learning | loss, sgd, cosine annealing, data loading | 20 min |
| lab 5 - back propagation | an autograd engine + the backward pass of every layer | 30 min |
| lab 6 - train it on memes | lenet-5, the training loop, accuracy on real memes | 25 min |
| bonus | [batching](addon_A_batching.ipynb) and [split-k](addon_B_split_k.ipynb) | |

## formula sheet (every formula in the lab, in one place)

notation: an image is $C \times H \times W$ (channels, rows, columns). a filter is $K \times K$. padding is $P$, stride is $S$. $dy$ always means $\partial L / \partial y$, the gradient coming back from the layer after.

**sizes**
- conv output: $\text{out} = \left\lfloor \dfrac{H + 2P - K}{S} \right\rfloor + 1$. in this lab $P = 0$ and $S = 1$ so its just $H - K + 1$ (32 → 28 with a 5x5 filter)
- max-pool output with a $p \times p$ window and stride $p$: $\lfloor H / p \rfloor$ (28 → 14)
- weights in a conv layer: $C_\text{out} \cdot C_\text{in} \cdot K \cdot K + C_\text{out}$ (the $+C_\text{out}$ is one bias per filter)
- weights in a fully connected layer: $n_\text{in} \cdot n_\text{out} + n_\text{out}$
- multiply-adds in a conv layer: $C_\text{out} \cdot OH \cdot OW \cdot C_\text{in} \cdot K \cdot K$

**forward**
- convolution: $y[k][i][j] = b[k] + \sum_{c}\sum_{u=0}^{K-1}\sum_{v=0}^{K-1} x[c][iS + u - P][jS + v - P] \cdot w[k][c][u][v]$. for us thats $x[c][i+u][j+v]$
- relu: $y = \max(0, x)$
- max-pool: $y[c][i][j] = \max_{u, v < p} \; x[c][ip + u][jp + v]$
- fully connected: $y[o] = b[o] + \sum_i W[o][i] \, x[i]$, in matrix form $y = Wx + b$
- softmax: $p_k = \dfrac{e^{z_k - m}}{\sum_j e^{z_j - m}}$ with $m = \max_j z_j$ (subtracting $m$ doesnt change anything, it just stops overflow)
- cross-entropy: $L = -\log p_y$ where $y$ is the right class. the safe way to compute it: $L = \log \sum_j e^{z_j - m} + m - z_y$

**memory and layouts**
- offset of a coordinate: $\text{offset} = \sum_i \text{coord}_i \cdot \text{stride}_i$
- row-major strides for shape $(C, H, W)$: $(HW,\; W,\; 1)$. column-major: $(1,\; C,\; CH)$
- pixel $(c, y, x)$ of a $C \times H \times W$ image sits at $cHW + yW + x$
- size $= \prod_i s_i$, cosize $= 1 + \sum_i (s_i - 1)\, d_i$
- im2col (the patches of a conv as a matrix): row $(oy, ox)$, column $(c, i, j)$, offset $= oy\,W + ox + cHW + iW + j$, so the layout is $((OH, OW), (C, K, K)) : ((W, 1), (HW, W, 1))$
- max-pool windows: $((C, H/p, W/p), (p, p)) : ((HW,\; pW,\; p), (W,\; 1))$
- conv as a matrix multiply: with $F$ = the filters as rows ($C_\text{out} \times CK^2$) and $X$ = the patches as rows ($OH \cdot OW \times CK^2$): $Y = F X^T$, then add $b[k]$ to row $k$

**backward**
- chain rule: $\dfrac{\partial L}{\partial x} = \dfrac{\partial L}{\partial y} \cdot \dfrac{\partial y}{\partial x}$, and if $x$ gets used twice its gradients add up
- relu: $dx = dy$ where $x > 0$, otherwise $0$
- fully connected: $dx = W^T dy$, $\quad dW[o][i] = dy[o]\, x[i]$, $\quad db = dy$
- max-pool: $dx = dy$ at the spot that won the max, $0$ everywhere else in that window
- softmax + cross-entropy together: $dz_k = p_k - [k = y]$ (the probabilities, minus 1 at the right answer)
- matrix multiply $C = A B^T$: $dA = dC \, B$, $\quad dB = dC^T A$
- convolution:
  - $db[k] = \sum_{i,j} dy[k][i][j]$
  - $dw[k][c][u][v] = \sum_{i,j} dy[k][i][j] \cdot x[c][i+u][j+v]$ (matrix form: $dF = dY \, X$)
  - $dx[c][h][w] = \sum_k \sum_{u,v} dy[k][h-u][w-v] \cdot w[k][c][u][v]$ over the $(h-u, w-v)$ that land inside $dy$. thats a "full" convolution of $dy$ with the filter flipped. matrix form: $dX = dY^T F$, then add every entry of $dX$ back onto the pixel it came from (scatter-add)

**learning**
- sgd: $w \leftarrow w - \text{lr} \cdot \dfrac{\partial L}{\partial w}$
- momentum: $v \leftarrow \mu v + g, \quad w \leftarrow w - \text{lr} \cdot v$
- cosine annealing: $\text{lr}(t) = \text{lr}_\min + \tfrac12 (\text{lr}_\max - \text{lr}_\min)\left(1 + \cos\dfrac{\pi t}{T}\right)$
- linear schedule: $\text{lr}(t) = \text{lr}_\max - (\text{lr}_\max - \text{lr}_\min)\, t / T$
- he init: starting weights $\sim \mathcal{N}\!\left(0,\; \sqrt{2 / \text{fan\_in}}\right)$ where fan_in is how many inputs go into one neuron ($C_\text{in} K^2$ for a conv)
- normalizing a pixel: $\hat v = (v / 255 - \text{mean}) / \text{std}$, per channel
- checking a gradient by nudging: $\dfrac{\partial L}{\partial w} \approx \dfrac{L(w + \epsilon) - L(w - \epsilon)}{2\epsilon}$
- accuracy = right guesses / total guesses

## python cheat sheet (everything the todos use)

**lists and loops**
- `range(n)` is 0, 1, ..., n-1. `range(a, b, step)` works too, like `range(0, 30, 10)` → 0, 10, 20
- `for x in xs:` goes over the values, `for i in range(len(xs)):` goes over the positions
- `xs.append(v)` adds to the end. `xs[i]` gets item i, `xs[-1]` is the last one, `len(xs)` is how many
- slicing: `xs[a:b]` is the items from a up to (not including) b. `xs[::k]` is every k-th item
- list comprehension: `[v * 2 for v in xs]` builds a new list. you can stack fors, `[v for row in grid for v in row]` flattens a grid (the left for is the outer loop)
- one line if/else: `a if cond else b`
- `zip(a, b)` walks two lists side by side: `for x, w in zip(xs, ws):`
- `enumerate(xs)` gives `(position, value)` pairs
- `sum(xs)`, `max(xs)`, `min(xs)`. `max(range(n), key=f)` gives the i where `f(i)` is biggest, thats an argmax
- `[0.0] * n` is a list of n zeros. careful: `[[0] * n] * n` makes n copies of the SAME row, use `[[0] * n for _ in range(n)]`
- `break` stops a loop early, `continue` skips to the next round

**tuples**
- `(3, 32, 32)` is a tuple, like a list you cant change. `(5,)` is a tuple with one thing in it (the comma matters)
- unpacking: `c, h, w = x.shape` or `(a, b), (c, d) = pair_of_pairs`
- `tuple(xs)` turns a list into a tuple. tuples can be dict keys, lists cant

**functions**
- `def f(a, b=2):` gives b a default. `return x, y` gives back a tuple
- `*args` collects extra arguments into a tuple, `**kwargs` collects name=value ones into a dict. `f(*xs)` spreads a list out into seperate arguments
- a function can call itself (recursion), like `relu_naive` calling `relu_naive` on each sub-list
- a function inside a function can use the outer one's variables, even later (a closure). the autograd is built on this
- `lambda x: x * 2` is a tiny unnamed function
- `isinstance(x, list)` checks if x is a list

**classes**
- `class Foo:` makes a new type. `def __init__(self, ...)` runs when you do `Foo(...)`. `self` is the object being made
- `self.data = data` stores something on the object, you get it back later with `obj.data`
- methods with double underscores are special: `__len__` makes `len(obj)` work, `__iter__` makes `for x in obj` work, `__call__` makes `obj(...)` work, `__getitem__` makes `obj[i]` work, `__repr__` is what prints
- `class Conv(Module):` means Conv gets all of Module's methods for free (inheritance)
- decorators: a line like `@property` right above a `def` changes how that function works. `@property` lets you write `t.layout` without the `()`
- you can stick a function onto a class later: `Tensor.backward = backward`

**generators**
- a function with `yield` in it hands out values one at a time instead of building a whole list. `for x, y in loader:` pulls them out one by one. thats how the data loader avoids decoding 35,000 images at once

**math and friends**
- `math.exp(x)`, `math.log(x)` (natural log), `math.sqrt(x)`, `math.cos(x)`, `math.pi`, `math.prod(xs)` (multiplies everything)
- `sumprod(a, b)` is `a[0]*b[0] + a[1]*b[1] + ...` (a dot product). its `math.sumprod` from python 3.12, done in c so its fast
- `operator.itemgetter(*idx)(xs)` grabs `xs[i]` for every i in idx at once (also fast)
- `random.gauss(0, s)` random number from a bell curve, `random.Random(seed)` your own random generator, `.shuffle(xs)` shuffles a list in place
- `time.perf_counter()` a clock for timing stuff

**files**
- `from pathlib import Path`. `Path("data")` is a path. `Path.cwd()` is the folder python is running in (for jupyter thats this notebook's folder)
- `/` joins paths: `DATA / "test.bin"`. `.parent` is one folder up
- `.exists()`, `.read_text()`, `.read_bytes()`, `.write_text(s)`
- a `bytes` object is a list of numbers 0-255. `raw[5]` gives an int, `raw[1:1025]` gives more bytes
- `json.dumps(obj)` / `json.loads(text)` turn lists and dicts into text and back

**jupyter**
- `%load file.py` pastes a file into the cell, `%run -i file.py` runs it right away

In [ ]:
# setup, plain python only
import json, math, operator, os, random, time
from pathlib import Path
try:
    from math import sumprod      # python 3.12+: sumprod(a, b) = a[0]*b[0] + a[1]*b[1] + ...  (a dot product done in c)
except ImportError:               # older python: same answer, just slower
    def sumprod(a, b):
        return sum(map(operator.mul, a, b))
import viz, checks                # viz draws pictures, checks grades your answers. neither is part of the network
random.seed(0)                    # same random numbers every time you run the notebook

In [ ]:
# catch-up: only if youre behind or want to skip ahead. delete the # on the last line and run this cell.
# it loads the answer to every lab. after that, dont re-run todo cells you havent finished,
# an unfinished cell would replace the working answer with the unfinished one.
# from babytorch import *

---
## lab 0 - the big picture

a neural network is a function with alot of knobs. **lenet-5** (yann lecun and friends, 1998. its 1989 ancestor read handwritten zip codes for the us postal service) takes the 3 x 32 x 32 = 3,072 numbers of an image and turns them into 10 scores, one per meme. it has 62,006 knobs, called *weights*. **training** means turning the knobs untill the right score wins. we use the modern version of it: relu and max-pooling instead of the old tanh and average pooling.

In [ ]:
viz.lenet_diagram()   # draws the network with the shape of every tensor

**convolution layers** slide small filters over the picture to find patterns (edges first, then eyes, sunglasses, toilets...). **max-pool layers** shrink the picture and keep the strongest signals. **fully connected layers** mix everything into the final 10 scores. you're going to build all of them.

### 0.1 meet the data
- **train**: 35,000 synthetic memes made from 457 real ones by a computer vision pipeline (crops, captions, emoji, blur...)
- **val**: 3,500 more synthetic memes for checking progress while training
- **test**: **150 real memes from the internet** that were never used for training. this is the real exam

every image is shrunk to 32 x 32 pixels, tiny, so plain python can keep up. the files use the cifar-10 format. each image is one record of 3,073 bytes:

| byte | what |
|---|---|
| 0 | the label, 0-9 |
| 1 ... 1024 | red values (0-255), row by row |
| 1025 ... 2048 | green |
| 2049 ... 3072 | blue |

In [ ]:
# given code - read it, then run it
#
# what this does: finds the data folder and reads a .bin file into python
# stuff in here:
#   RECORD = 1 + 3 * 32 * 32     how many bytes one image takes up (1 label byte + 3,072 pixels)
#   Path(...)                    a path to a file or folder (from pathlib). paths join with /, like DATA / "test.bin"
#   Path.cwd()                   the folder python is running in. in jupyter thats the folder this notebook is in (src/)
#   Path(__file__).resolve()     where this code's own file is. a notebook doesnt have one, so it uses Path.cwd()
#   "__file__" in globals()      true when the code runs from a .py file instead of a notebook
#   here.parent                  one folder up, so here.parent / "data" is ../data
#   os.environ.get("MEME10_DATA")   reads an environment varible if you set one, otherwise gives None
#   .exists()                    true if the file is really there
#   .read_text().split()         reads a whole file as text and splits it on spaces/newlines (one class per line)
#   .read_bytes()                reads a whole file as bytes (a list of numbers 0-255)
#   raw[i]                       one byte, comes out as an int (thats the label)
#   raw[i + 1:i + RECORD]        the 3,072 pixel bytes right after it
#   range(0, len(raw), RECORD)   0, 3073, 6146, ... where every record starts
#   raise FileNotFoundError(...) stops everything with an error message
#
RECORD = 1 + 3 * 32 * 32  # one image in a .bin file = 1 label byte + 3,072 pixel bytes (red plane, green plane, blue plane)


def find_data():
    """finds the folder with the meme dataset (the data/ folder next to src/)"""
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    for candidate in [os.environ.get("MEME10_DATA"), here.parent / "data", Path.cwd().parent / "data",
                      Path.cwd() / "data"]:
        if candidate and (Path(candidate) / "classes.txt").exists():
            return Path(candidate)
    raise FileNotFoundError("cant find the data/ folder. put it next to src/ or set MEME10_DATA=/path/to/data")


DATA = find_data()
CLASSES = (DATA / "classes.txt").read_text().split()


def load_bin(path):
    """reads a .bin file into a list of (label, pixels) pairs. pixels is 3,072 bytes, each one 0..255"""
    raw = Path(path).read_bytes()
    return [(raw[i], raw[i + 1:i + RECORD]) for i in range(0, len(raw), RECORD)]

In [ ]:
test = load_bin(DATA / "test.bin")          # a list of (label, pixels) pairs, 150 of them
label, pixels = test[0]                      # unpack the first pair
print(len(test), "test images. image 0 has label", label, "=", CLASSES[label])
print("its first 8 red values (top left corner):", list(pixels[:8]))   # list() turns bytes into ints so they print nice
viz.show_images([(px, CLASSES[lab]) for lab, px in test[12::15]], scale=3, title="one real test meme per class (32 x 32)")

pixel (row `y`, column `x`) of channel `c` sits at **`c*1024 + y*32 + x`** inside `pixels`. remember this one, its basicly the whole idea of lab 2.

try it: whats the *blue* value at row 31, column 0 of image 0? change `c, y, x` and run it again.

In [ ]:
c, y, x = 1, 5, 7                            # channel 1 is green
print(f"channel {c}, row {y}, column {x}:", pixels[c * 1024 + y * 32 + x])

### 0.2 shapes and knobs
a k x k filter with no padding fits into a row of `size` pixels at `size - k + 1` spots. a p x p max-pool keeps one value per p x p window. write four tiny functions, then use them to work out lenet-5's shapes. (all the formulas are on the formula sheet at the top)

In [ ]:
# answer (same as solutions/shapes.py)
def conv_out(size, k):
    """how wide the output is after a k x k conv with no padding and stride 1"""
    return size - k + 1


def pool_out(size, p):
    """how wide the output is after p x p max-pooling (the windows dont overlap)"""
    return size // p


def conv_params(c_in, c_out, k):
    """weights + biases in a conv layer: c_out filters, each one c_in x k x k, plus one bias per filter"""
    return c_out * c_in * k * k + c_out


def linear_params(n_in, n_out):
    """weights + biases in a fully connected layer: a weight for every (input, output) pair, plus a bias per output"""
    return n_in * n_out + n_out

In [ ]:
checks.shapes(conv_out, pool_out, conv_params, linear_params)

In [ ]:
# walks thru lenet-5 layer by layer using your functions
H = 32
print(f"input             3 x {H} x {H}")
H = conv_out(H, 5); print(f"conv 5x5 (6)      6 x {H} x {H}     knobs: {conv_params(3, 6, 5):>6,}")
H = pool_out(H, 2); print(f"max-pool 2x2      6 x {H} x {H}")
H = conv_out(H, 5); print(f"conv 5x5 (16)    16 x {H} x {H}     knobs: {conv_params(6, 16, 5):>6,}")
H = pool_out(H, 2); print(f"max-pool 2x2     16 x {H} x {H}  -> flatten to {16 * H * H} numbers")
print(f"fully connected  {16 * H * H} -> 120          knobs: {linear_params(16 * H * H, 120):>6,}")
print(f"fully connected  120 -> 84           knobs: {linear_params(120, 84):>6,}")
print(f"fully connected  84 -> 10            knobs: {linear_params(84, 10):>6,}")

talk about it: which layer owns most of the knobs? which layers do most of the *multiplying*? (its not the same ones)

### answer: knobs vs multiplies

- **most knobs:** the first fully connected layer, 400 -> 120, has 400*120 + 120 = **48,120** parameters, about 78% of all 62,006. the second conv has 2,416, the first conv 456, and the last two FC layers 10,164 and 850.
- **most multiplying:** the convolutions. conv1 produces 6*28*28 = 4,704 outputs, each from a 3*5*5 = 75-pixel patch, so about 352,800 multiplies; conv2 does 16*10*10 * 6*5*5 = 240,000. the FC layers do 48,000 + 10,080 + 840. max-pool only compares, it multiplies nothing.
- so the layer with the most weights is not the layer doing the most work: the big FC layer stores most knobs, while the early convolutions burn most of the compute.

---
## lab 1 - layers, the slow obvious way

first we write every layer the most direct way possible. an image is a list of channels, a channel is a list of rows, a row is a list of numbers: `image[c][y][x]`. lots of nested loops. its correct and easy to read and slow, thats fine for now.

In [ ]:
# given code - read it, then run it
#
# what this does: turns the flat list of 3,072 pixels into image[c][y][x] and scales them to 0..1
# stuff in here:
#   C=3, H=32, W=32              default values, you only pass them if your picture is a different size
#   [[[... for x in range(W)] for y in range(H)] for c in range(C)]
#                                a comprehension inside a comprehension inside a comprehension. the outside one
#                                makes the channels, the middle one the rows, the inside one the numbers in a row
#   c * H * W + y * W + x        the pixel formula from lab 0 (with H = W = 32 thats c*1024 + y*32 + x)
#   / 255                        bytes go 0-255, this squishes them to 0..1
#
def to_nested(pixels, C=3, H=32, W=32):
    """flat CHW pixels (bytes or numbers) -> image[c][y][x], scaled to 0..1"""
    return [[[pixels[c * H * W + y * W + x] / 255 for x in range(W)] for y in range(H)] for c in range(C)]

### 1.1 convolution
a **filter** (or *kernel*) is a small block of weights, here C x K x K. put it on top of the image, multiply every pair of numbers that line up, add them all up and add a bias: thats one output pixel. slide one step right and do it again. each filter makes one **feature map**, so a layer with 6 filters makes 6 maps.

$$\text{out}[k][y][x] = \text{bias}[k] + \sum_{c}\sum_{i}\sum_{j} \text{image}[c][y+i][x+j]\cdot\text{kernel}[k][c][i][j]$$

the output is smaller than the input, $OH = H - K + 1$ and $OW = W - K + 1$, becuase the filter cant hang off the edge. (with padding $P$ and stride $S$ its $\lfloor (H + 2P - K)/S \rfloor + 1$. we always use $P = 0$, $S = 1$)

```
 image rows y..y+2      filter          one output number
  3  1  0  2  5         1  0 -1
  2  8  1  0  1    x    2  0 -2    ->   3*1 + 1*0 + 0*(-1) + 2*2 + 8*0 + 1*(-2) + ...
  0  1  4  2  2         1  0 -1
```

In [ ]:
# answer (same as solutions/conv2d_naive.py)
def conv2d_naive(image, kernels, biases):
    """image[c][y][x] is C x H x W, kernels[k][c][i][j] is Cout x C x K x K, biases[k] is one number per filter.
    gives back out[k][y][x], one feature map per filter.

    every output pixel is one dot product: lay the filter over the image, multiply what lines up, add it all up"""
    C, H, W = len(image), len(image[0]), len(image[0][0])
    K = len(kernels[0][0])
    OH, OW = H - K + 1, W - K + 1
    out = []
    for k in range(len(kernels)):  # one feature map per filter
        fmap = []
        for y in range(OH):
            row = []
            for x in range(OW):
                total = biases[k]
                for c in range(C):
                    for i in range(K):
                        for j in range(K):
                            total += image[c][y + i][x + j] * kernels[k][c][i][j]
                row.append(total)
            fmap.append(row)
        out.append(fmap)
    return out

In [ ]:
checks.conv2d_naive(conv2d_naive)

### 1.2 relu, max-pool, flatten
- **relu**: `max(0, v)`. keep the positive evidence, throw away the negative. without it the whole network would collapse into one big linear function (stacking linear layers just gives you another linear layer)
- **max-pool**: keep the biggest number in every p x p window. the picture gets half as wide and half as tall, and the network cares less about *exactly* where something is
- **flatten**: line everything up into one long list for the fully connected layers

In [ ]:
# answer (same as solutions/relu_naive.py)
def relu_naive(x):
    """max(0, v) for every number v in a (nested) list"""
    if isinstance(x, list):
        return [relu_naive(v) for v in x]
    return x if x > 0 else 0.0

In [ ]:
checks.relu_naive(relu_naive)

In [ ]:
# answer (same as solutions/maxpool2d_naive.py)
def maxpool2d_naive(fmaps, p=2):
    """keeps the biggest value in every p x p window (stride p) of every feature map"""
    out = []
    for fmap in fmaps:
        H, W = len(fmap), len(fmap[0])
        out.append([[max(fmap[y * p + i][x * p + j] for i in range(p) for j in range(p))
                     for x in range(W // p)]
                    for y in range(H // p)])
    return out

In [ ]:
checks.maxpool2d_naive(maxpool2d_naive)

In [ ]:
# answer (same as solutions/flatten_naive.py)
def flatten_naive(fmaps):
    """C x H x W nested lists -> one flat list (channel by channel, row by row)"""
    return [v for fmap in fmaps for row in fmap for v in row]

In [ ]:
checks.flatten_naive(flatten_naive)

### 1.3 watch a convolution work
real networks *learn* their filters. here are three classic hand made ones on a trollface. bright means the filter found its pattern there.

In [ ]:
label, pixels = test[90]                                # test image 90 is a trollface
img = to_nested(pixels)
vertical   = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]      # left to right changes = vertical edges
horizontal = [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]      # top to bottom changes = horizontal edges
blur       = [[1 / 9] * 3 for _ in range(3)]           # the average of a 3x3 neighbourhood
kernels = [[f, f, f] for f in (vertical, horizontal, blur)]   # the same 3x3 pattern on red, green and blue
maps = conv2d_naive(img, kernels, [0.0, 0.0, 0.0])            # 3 filters -> 3 maps of 30 x 30
viz.show_image(pixels, caption=CLASSES[label], scale=4)
viz.show_maps(flatten_naive(maps), 3, 30, 30, labels=["vertical edges", "horizontal edges", "blur"], title="conv2d_naive")
pooled = maxpool2d_naive(relu_naive(maps))
viz.show_maps(flatten_naive(pooled), 3, 15, 15, scale=8, labels=["vertical", "horizontal", "blur"],
              title="after relu + 2x2 max-pool: a quarter of the numbers, same story")

try it: make up your own 3 x 3 filter (`[[0, 1, 0], [1, -4, 1], [0, 1, 0]]` finds outlines) or pick a different picture, `test[0]` ... `test[149]`.

### 1.4 fully connected layer and softmax
a **fully connected** (linear) layer: every output looks at every input. output `o` is a dot product of the input with row `o` of the weight matrix, plus a bias:

$$\text{out}[o] = b[o] + \sum_i W[o][i]\cdot x[i]$$

**softmax** turns the final 10 scores (*logits*) into probabilities that add up to 1: $p_k = e^{z_k} / \sum_j e^{z_j}$. subtract the biggest score first or `math.exp(1000)` overflows.

In [ ]:
# answer (same as solutions/linear_naive.py)
def linear_naive(x, weights, biases):
    """fully connected layer: out[o] = biases[o] + the sum over i of weights[o][i] * x[i]"""
    out = []
    for o in range(len(weights)):
        total = biases[o]
        for i in range(len(x)):
            total += weights[o][i] * x[i]
        out.append(total)
    return out

In [ ]:
checks.linear_naive(linear_naive)

In [ ]:
# answer (same as solutions/softmax.py)
def softmax(logits):
    """turns scores into probabilities that add up to 1: exp(z) / sum(exp(z))"""
    m = max(logits)  # taking away the max doesnt change the answer, it just stops exp() from overflowing
    exps = [math.exp(z - m) for z in logits]
    total = sum(exps)
    return [e / total for e in exps]

In [ ]:
checks.softmax(softmax)

### 1.5 the whole lenet-5, slowly
stack your layers: conv → relu → pool → conv → relu → pool → flatten → linear → relu → linear → relu → linear → softmax. the weights are random for now (**he init**: random numbers with standard deviation $\sqrt{2/\text{fan\_in}}$, the same `Init::He` cudagrad uses), so expect a useless, almost even guess.

In [ ]:
# given code - read it, then run it
#
# what this does: makes random weights for lenet-5 and runs your lab 1 layers in the right order
# stuff in here:
#   random.gauss(0.0, std)       a random number from a bell curve centered on 0, `std` is how spread out it is
#   math.sqrt(2.0 / fan_in)      "he init": layers with more inputs get smaller starting weights so nothing blows up
#   for _ in range(n)            _ is a name for a loop varible you dont actually use
#   nest(flat_list, shape)       recursion: cut the list into shape[0] pieces and nest every piece the same way
#   LENET_SHAPES = {...}         a dict: name -> shape. "conv1.w" is the first conv's filters, "conv1.b" its biases
#   params.items()               every (name, value) pair in a dict
#   math.prod(shape)             how many numbers fit in that shape (6 * 3 * 5 * 5 = 450)
#   name.endswith(".w")          string check, true for weights
#   a if cond else b             weights get he init, biases get zeros
#   {name: ... for name, v in params.items()}   a dict comprehension, builds a new dict
#
def he_init(n, fan_in):
    """n random weights from a bell curve with std sqrt(2 / fan_in) ("he" init, same as cudagrad's Init::He)"""
    std = math.sqrt(2.0 / fan_in)
    return [random.gauss(0.0, std) for _ in range(n)]


def nest(flat_list, shape):
    """turns a flat list into nested lists: nest(list(range(6)), (2, 3)) -> [[0, 1, 2], [3, 4, 5]]"""
    if len(shape) == 1:
        return list(flat_list)
    step = len(flat_list) // shape[0]
    return [nest(flat_list[i * step:(i + 1) * step], shape[1:]) for i in range(shape[0])]


LENET_SHAPES = {  # name -> shape of every weight and bias in lenet-5
    "conv1.w": (6, 3, 5, 5), "conv1.b": (6,),
    "conv2.w": (16, 6, 5, 5), "conv2.b": (16,),
    "fc1.w": (120, 400), "fc1.b": (120,),
    "fc2.w": (84, 120), "fc2.b": (84,),
    "fc3.w": (10, 84), "fc3.b": (10,),
}


def init_lenet_params(seed=0):
    """random lenet-5 parameters as flat lists (weights use he init, biases start at 0)"""
    random.seed(seed)
    params = {}
    for name, shape in LENET_SHAPES.items():
        n = math.prod(shape)
        params[name] = he_init(n, n // shape[0]) if name.endswith(".w") else [0.0] * n
    return params


def lenet_forward_naive(image, params):
    """lenet-5's forward pass built out of the lab 1 layers. gives back 10 class probabilities"""
    P = {name: nest(v, LENET_SHAPES[name]) for name, v in params.items()}
    x = maxpool2d_naive(relu_naive(conv2d_naive(image, P["conv1.w"], P["conv1.b"])))   # 6 x 14 x 14
    x = maxpool2d_naive(relu_naive(conv2d_naive(x, P["conv2.w"], P["conv2.b"])))       # 16 x 5 x 5
    x = flatten_naive(x)                                                                # 400
    x = relu_naive(linear_naive(x, P["fc1.w"], P["fc1.b"]))                             # 120
    x = relu_naive(linear_naive(x, P["fc2.w"], P["fc2.b"]))                             # 84
    return softmax(linear_naive(x, P["fc3.w"], P["fc3.b"]))                             # 10

In [ ]:
params = init_lenet_params(seed=0)                     # a dict of flat lists, one per weight/bias
t = time.perf_counter()                                # start the clock
probs = lenet_forward_naive(to_nested(pixels), params)
naive_ms = (time.perf_counter() - t) * 1000            # stop the clock, in milliseconds
viz.show_bars(probs, CLASSES, f"untrained lenet-5 looking at a {CLASSES[label]}")
print(f"one forward pass: {naive_ms:.0f} ms")
print(f"training = forward + backward (~3x a forward) for 35,000 images x 4 epochs = "
      f"{3 * naive_ms * 35000 * 4 / 3.6e6:.1f} hours")

thats way too slow. real libraries never loop over nested lists. they keep a tensor in **one flat block of memory** plus a recipe for finding stuff in it. thats next.

---
## lab 2 - how data is stored

computer memory is one long row of numbers. a 3 x 32 x 32 image is 3,072 numbers in a row, and in lab 0 you found pixel (c, y, x) at `c*1024 + y*32 + x`. that recipe is called a **layout**.

nvidia's **cute** library (part of cutlass, the code behind the fastest matrix multiplies on gpus) describes every layout with two tuples:

- a **shape**: how many steps there are in each direction, like `(3, 32, 32)`
- a **stride**: how far thru memory one step goes, like `(1024, 32, 1)`

and it prints them as `shape:stride`, here `(3,32,32):(1024,32,1)`. the memory **offset** of a coordinate is just the coordinate times the stride:

$$\text{offset}(c, y, x) = c\cdot 1024 + y\cdot 32 + x\cdot 1$$

thats really the whole idea. transposes, slices, broadcasting, flattening, even convolution patches are all just a different shape and stride over the same memory. **no data gets copied.**

> real cute vs this lab: in cute (c++) you'd write `make_layout(make_shape(3,32,32), make_stride(1024,32,1))`, and in nvidia's python cute dsl `cute.make_layout((3, 32, 32), stride=(1024, 32, 1))` (that one needs an nvidia gpu). we copy cute's notation and names (`size`, `cosize`, `print_layout`, `zipped_divide`) with two differences: our default strides are **row-major** like python and pytorch (cute defaults to column-major), and a single number counts thru a layout row-major (cute counts the first mode fastest).

### 2.1 coordinates → offsets
shapes can be **nested**: `((28, 28), (3, 5, 5))` is a 2-D layout whose rows are themselves 2-D and whose columns are 3-D. these helpers flatten nested tuples so the math only ever sees flat ones.

In [ ]:
# given code - read it, then run it
#
# what this does: little helpers for nested tuples like ((28, 28), (3, 5, 5)), so the layout math only ever
# sees flat ones
# stuff in here:
#   isinstance(t, tuple)         true if t is a tuple
#   tuple(x for item in t for x in flat(item))   recursion + a generator: flatten every item, glue them together
#   iter(values), next(values)   an iterator hands out the next value every time you call next() on it
#   def rebuild(t): inside unflat   a function inside a function, it can use `values` from the outer one
#   i % n, i //= n               remainder and floor division. its how you split a number into "digits"
#                                (125 -> 1, 2, 5 when every digit goes up to 10)
#   ",".join(parts)              glues strings together with commas in between
#
def flat(t):
    """flattens a nested tuple: ((28, 28), (3, 5, 5)) -> (28, 28, 3, 5, 5)"""
    if isinstance(t, tuple):
        return tuple(x for item in t for x in flat(item))
    return (t,)


def unflat(values, like):
    """puts flat values back into the nesting of `like`: unflat((1, 2, 3), ((0, 0), 0)) -> ((1, 2), 3)"""
    values = iter(values)

    def rebuild(t):
        return tuple(rebuild(x) for x in t) if isinstance(t, tuple) else next(values)
    return rebuild(like)


def idx2crd(i, shape):
    """one integer index -> a coordinate, counting row-major: idx2crd(5, (2, 3)) -> (1, 2)"""
    if not isinstance(shape, tuple):
        return i
    crd = []
    for s in reversed(shape):
        n = math.prod(flat(s))
        crd.append(idx2crd(i % n, s))
        i //= n
    return tuple(reversed(crd))


def fmt(t):
    """cute style text for a (nested) tuple: (3, (5, 5)) -> '(3,(5,5))'"""
    return "(" + ",".join(fmt(x) for x in t) + ")" if isinstance(t, tuple) else str(t)

In [ ]:
# answer (same as solutions/crd2idx.py)
def crd2idx(coord, stride):
    """the heart of a layout: offset = coord[0]*stride[0] + coord[1]*stride[1] + ... (both are flat tuples)"""
    return sum(c * d for c, d in zip(coord, stride))

In [ ]:
checks.crd2idx(crd2idx)

### 2.2 row-major and column-major
when you leave the stride out, a layout is **compact**: no gaps and no overlaps. there are two natural ways to do that. **row-major**: the last coordinate moves fastest, so rows sit together in memory (python, c and pytorch do this, cute calls it `LayoutRight`). **column-major**: the first coordinate moves fastest (fortran, matlab, and cute's default `LayoutLeft`).

In [ ]:
# answer (same as solutions/strides.py)
def row_major(shape):
    """compact row-major strides (cute calls it LayoutRight). the last mode moves fastest. (3, 32, 32) -> (1024, 32, 1)"""
    strides, step = [], 1
    for s in reversed(shape):
        strides.append(step)
        step *= s
    return tuple(reversed(strides))


def col_major(shape):
    """compact column-major strides (cute calls it LayoutLeft). the first mode moves fastest. (3, 32, 32) -> (1, 3, 96)"""
    strides, step = [], 1
    for s in shape:
        strides.append(step)
        step *= s
    return tuple(strides)

In [ ]:
checks.strides(row_major, col_major)

now the `Layout` class itself. its mostly bookkeeping around your `crd2idx` and `row_major`. the important part: a layout is **a function**. you call it with a coordinate and it gives back an offset.

In [ ]:
# given code - read it, then run it
#
# what this does: the Layout class. a layout is a shape and a stride, and calling it turns a coordinate into
# a memory offset
# stuff in here:
#   class Layout:                a new type of object. Layout((3, 32, 32)) makes one
#   def __init__(self, ...)      runs when you make one. self is the new layout. stride=None means its optional
#   self.shape = ...             stores stuff on the object so the other methods can use it later
#   ... in flat(...)             the ... (its called Ellipsis) check is for the fill in the blank todos later
#   raise ValueError(...)        complains if the shape and stride dont have the same structure
#   def __call__(self, *coord)   lets you call the object like a function: layout(1, 5, 7).
#                                *coord collects all the arguments into one tuple
#   def __getitem__(self, i)     makes layout[0] work, it gives one mode as its own layout
#   def __len__(self)            makes len(layout) work: how many modes (the "rank")
#   def __eq__ / __hash__        makes == work and lets layouts be dict keys
#   def __repr__(self)           what you see when you print it, the cute style shape:stride
#   f"{a}:{b}"                   an f-string, puts values into a string
#   make_layout(L1, L2)          glues two layouts into one that has two modes
#
class Layout:
    """a cute style layout: a shape and a stride with the same (maybe nested) structure.

    its a function from coordinates to memory offsets:  Layout((3, 32, 32), (1024, 32, 1))(1, 5, 7) == 1191.
    it prints the cute way, shape:stride  ->  (3,32,32):(1024,32,1)
    leave out the stride and you get compact row-major strides"""

    def __init__(self, shape, stride=None):
        self.shape = shape if isinstance(shape, tuple) else (shape,)
        if stride is None:
            stride = unflat(row_major(flat(self.shape)), self.shape)
        self.stride = stride if isinstance(stride, tuple) else (stride,)
        if ... in flat(self.shape) + flat(self.stride):  # a fill in the blank todo that isnt filled in yet
            raise NotImplementedError("todo: replace every ... in the layout with a number")
        if unflat(flat(self.stride), self.shape) != self.stride:
            raise ValueError(f"shape {fmt(self.shape)} and stride {fmt(self.stride)} need the same structure")

    def __call__(self, *coord):
        """coordinate -> offset. each coordinate can be a tuple (one entry per sub-mode) or one integer"""
        if len(coord) == 1 and len(self.shape) > 1:   # one integer for the whole layout: count row-major
            coord = idx2crd(coord[0], self.shape)
        return crd2idx(_flat_coord(coord, self.shape), flat(self.stride))

    def __getitem__(self, i):
        """mode i as its own layout:  Layout((3, 32, 32))[0] == Layout(3, 1024)"""
        return Layout(self.shape[i], self.stride[i])

    def __len__(self):
        return len(self.shape)  # the rank, how many modes

    def __eq__(self, other):
        return isinstance(other, Layout) and (self.shape, self.stride) == (other.shape, other.stride)

    def __hash__(self):
        return hash((self.shape, self.stride))

    def __repr__(self):
        if len(self.shape) == 1:
            return f"{fmt(self.shape[0])}:{fmt(self.stride[0])}"
        return f"{fmt(self.shape)}:{fmt(self.stride)}"


def _flat_coord(coord, shape):
    """turns a coordinate (where some entries can be one integer for a nested mode) into a flat tuple"""
    if not isinstance(shape, tuple):
        return (coord,)
    if not isinstance(coord, tuple):
        coord = idx2crd(coord, shape)
    if len(coord) != len(shape):
        raise ValueError(f"coordinate {coord} doesnt match shape {fmt(shape)}")
    return tuple(x for c, s in zip(coord, shape) for x in _flat_coord(c, s))


def make_layout(*layouts):
    """glues layouts together into one layout whose modes are their modes (cute's make_layout(a, b))"""
    return Layout(tuple(L.shape if len(L) > 1 else L.shape[0] for L in layouts),
                  tuple(L.stride if len(L) > 1 else L.stride[0] for L in layouts))

In [ ]:
image = Layout((3, 32, 32))          # no stride given -> row-major
print(image)                         # (3,32,32):(1024,32,1)
print(image(1, 5, 7))                # channel 1, row 5, column 7 -> 1*1024 + 5*32 + 7
print(pixels[image(1, 5, 7)])        # a layout + memory = a tensor. this is the actual pixel

### 2.3 size and cosize
- **size**: how many coordinates a layout has (multiply the shape together)
- **cosize**: how much memory it reaches (1 + the offset of its last coordinate)

for a compact layout the two are the same. soon they wont be.

In [ ]:
# answer (same as solutions/size_cosize.py)
def size(layout):
    """how many coordinates the layout has, the product of its shape"""
    return math.prod(flat(layout.shape))


def cosize(layout):
    """how much memory it reaches: 1 + the offset of its last coordinate (cute: A(size(A) - 1) + 1)"""
    return 1 + sum((s - 1) * d for s, d in zip(flat(layout.shape), flat(layout.stride)))

In [ ]:
checks.size_cosize(size, cosize)

In [ ]:
# given code - read it, then run it
#
# what this does: prints a layout with two modes as a table of offsets (cute has a function with the same name)
# stuff in here:
#   [[layout(i, j) for j in range(N)] for i in range(M)]   every offset in the table
#   f"{v:>{w}}"                  right-aligns v in a space w characters wide
#   "-" * (w + 2)                repeats a string
#   "+".join(...)                glues pieces together with + in between
#
def print_layout(layout):
    """prints a rank-2 layout as a table of offsets, like cute's print_layout"""
    if len(layout) == 1:
        layout = make_layout(Layout(1, 0), layout)
    if len(layout) != 2:
        raise ValueError("print_layout needs a rank-2 layout (rows, columns)")
    M, N = size(layout[0]), size(layout[1])
    cells = [[layout(i, j) for j in range(N)] for i in range(M)]
    w = max(len(str(v)) for row in cells for v in row)
    w = max(w, len(str(N - 1)))
    line = "    +" + "+".join("-" * (w + 2) for _ in range(N)) + "+"
    print(layout)
    print("     " + " ".join(f" {j:>{w}} " for j in range(N)))
    print(line)
    for i, row in enumerate(cells):
        print(f"{i:>3} |" + "|".join(f" {v:>{w}} " for v in row) + "|")
        print(line)

In [ ]:
print_layout(Layout((4, 8), (8, 1)))   # row-major: walk along a row and the offset goes up by 1
print_layout(Layout((4, 8), (1, 4)))   # column-major: the SAME 32 numbers in a different order

### 2.4 views: same memory, new layout
most "operations" on tensors dont touch the numbers at all, they only change the layout.

- **transpose**: swap the two modes. `(4,8):(8,1)` becomes `(8,4):(1,8)`
- **flatten**: `(16,5,5):(25,5,1)` and `400:1` reach the exact same offsets in the same order
- **slice**: one channel of the image is `(32,32):(32,1)` starting at offset 1024
- **broadcast**: a stride of **0** means that direction doesnt move. `(4,3):(0,1)` is 4 rows that all read the same 3 numbers

In [ ]:
# answer (same as solutions/transpose.py)
def transpose(layout):
    """swaps the two modes of a rank-2 layout. no data moves, only the layout changes"""
    (s0, s1), (d0, d1) = layout.shape, layout.stride
    return Layout((s1, s0), (d1, d0))

In [ ]:
checks.transpose(transpose)

In [ ]:
# given code - read it, then run it
#
# what this does: lists every offset a layout points at, in row-major order, and remembers the answer
# stuff in here:
#   _OFFSETS = {}                a dict used as a cache, so the same layout only gets worked out once
#   key = (layout.shape, layout.stride)   tuples can be dict keys (lists cant)
#   [o + i * d for o in offs for i in range(s)]
#                                start with [0]. a mode with size s and stride d turns every offset o into
#                                o, o + d, o + 2d, ... do that for every mode and youre done
#
_OFFSETS = {}


def offsets(layout):
    """every offset a layout points at, in row-major coordinate order (kept in a cache, it only depends on the layout).

    built one mode at a time: start from [0]. a mode with size s and stride d turns every offset o
    into o, o + d, o + 2d, ..., o + (s-1)d"""
    key = (layout.shape, layout.stride)
    if key not in _OFFSETS:
        offs = [0]
        for s, d in zip(flat(layout.shape), flat(layout.stride)):
            offs = [o + i * d for o in offs for i in range(s)]
        _OFFSETS[key] = offs
    return _OFFSETS[key]

In [ ]:
print_layout(transpose(Layout((2, 3))))                              # a 2x3 matrix read column by column
print("flatten is free:", offsets(Layout((16, 5, 5))) == offsets(Layout(400)))
print("green(5, 7) =", pixels[1024 + Layout((32, 32))(5, 7)])        # a slice is a smaller layout + a starting offset
viz.show_layout(Layout((4, 3), (0, 1)), title="broadcast (4,3):(0,1), same colour = same spot in memory")

### 2.5 gather: read memory thru a layout
to compute with a view you read the numbers it points at, in order. thats a **gather**.

In [ ]:
# answer (same as solutions/gather_simple.py)
def gather(storage, layout):
    """reads memory thru a layout: one value for every coordinate, in row-major order"""
    return [storage[o] for o in offsets(layout)]

In [ ]:
checks.gather_simple(gather)

In [ ]:
# given code - read it, then run it
#
# what this does: the same job as your gather, way faster. it replaces yours for the rest of the notebook
# stuff in here:
#   operator.itemgetter(*offs)   makes a function that grabs storage[o] for every o at once, and it runs in c.
#                                *offs spreads the list out into seperate arguments
#   _GETTERS                     a cache again, one getter per layout
#   lambda s, o=offs[0]: (s[o],) a tiny one line function for the one offset case (itemgetter with one index
#                                gives back a plain value instead of a tuple)
#
_GETTERS = {}


def gather(storage, layout):  # replaces the simple version: same answer, much faster
    """reads memory thru a layout (fast version).

    the offsets only depend on the layout so we work them out once and keep an operator.itemgetter,
    which grabs all of them in one call that runs in c. (cutlass precomputes its offsets too)"""
    key = (layout.shape, layout.stride)
    getter = _GETTERS.get(key)
    if getter is None:
        offs = offsets(layout)
        getter = operator.itemgetter(*offs) if len(offs) > 1 else (lambda s, o=offs[0]: (s[o],))
        _GETTERS[key] = getter
    return getter(storage)

### 2.6 the big one: convolution patches as a layout (im2col)
look at your `conv2d_naive` again. for output pixel `(oy, ox)` it reads a **patch**: `image[c][oy+i][ox+j]` for every `c, i, j`. thats 3 x 5 x 5 = 75 numbers for lenet's first layer, at 28 x 28 = 784 output pixels. put every patch in a row and you get a 784 x 75 matrix, which people call the **im2col** matrix.

copying it out would cost 58,800 numbers. instead, write it as a layout over the image's memory. pixel `(c, oy+i, ox+j)` lives at

$$c\cdot HW + (oy+i)\cdot W + (ox+j) \;=\; \underbrace{oy\cdot W + ox\cdot 1}_{\text{which patch}} \;+\; \underbrace{c\cdot HW + i\cdot W + j\cdot 1}_{\text{where in the patch}}$$

so the rows are the mode `(OH, OW)` and the columns are the mode `(C, K, K)`. read the strides right off that formula.

In [ ]:
# answer (same as solutions/im2col_layout.py)
def im2col_layout(C, H, W, K):
    """the patches of a K x K convolution as a layout over a C x H x W image stored row-major (strides H*W, W, 1).

    row p    = an output pixel (oy, ox)        -> mode (OH, OW)
    column j = a spot (c, i, j) in the patch    -> mode (C, K, K)
    pixel (c, oy + i, ox + j) lives at  c*H*W + (oy + i)*W + (ox + j)
                                     = oy*W + ox*1  +  c*H*W + i*W + j*1"""
    OH, OW = H - K + 1, W - K + 1
    return Layout(((OH, OW), (C, K, K)),
                  ((W, 1), (H * W, W, 1)))

In [ ]:
checks.im2col_layout(im2col_layout)

In [ ]:
L = im2col_layout(1, 4, 4, 2)        # a tiny 1 x 4 x 4 image with 2 x 2 patches
print(L, "  size", size(L), "  cosize", cosize(L))
viz.show_layout(L, title=f"im2col_layout(1, 4, 4, 2) = {L}: 9 patches x 4 numbers out of only 16 memory spots")

**size 36 but cosize 16**: the layout reads the same memory spot more than once (same colour = same spot) becuase neighbouring patches overlap. the patch matrix is real enough to compute with but it never got built.

same trick on lenet's first layer:

In [ ]:
L1 = im2col_layout(3, 32, 32, 5)
print(L1)
print(f"a {size(L1[0])} x {size(L1[1])} patch matrix = {size(L1):,} numbers, read out of {cosize(L1):,} numbers of memory")

### 2.7 max-pool windows as a layout
same game: a layout where every row is a 2 x 2 window and the columns are the 4 spots inside it.

In [ ]:
# answer (same as solutions/pool_layout.py)
def pool_layout(C, H, W, P):
    """the P x P max-pool windows of a C x H x W image, one window per row.

    row    = a window (c, oy, ox)    -> mode (C, H//P, W//P), the window starts at c*H*W + oy*P*W + ox*P
    column = a spot (i, j) inside it  -> mode (P, P), offset i*W + j"""
    return Layout(((C, H // P, W // P), (P, P)),
                  ((H * W, P * W, P), (W, 1)))

In [ ]:
checks.pool_layout(pool_layout)

In [ ]:
viz.show_layout(pool_layout(1, 4, 4, 2), title="pool_layout(1, 4, 4, 2): 4 windows, no overlap")

> in cute you dont write that one by hand, you *divide* the image layout by a tile. `zipped_divide(layout, (1, 2, 2))` splits every mode into (inside the tile, which tile) and groups them. cutlass uses exactly this to hand out tiles of a matrix to gpu threads.

In [ ]:
# given code - read it, then run it
#
# what this does: cuts every mode of a layout into tiles, a simple version of cute's zipped_divide
# stuff in here:
#   zip(layout.shape, layout.stride, tiler)   walks three tuples together
#   s % t                        remainder. if its not 0 the tile doesnt fit into the mode evenly
#   the rule: a mode with size s and stride d, tiled by t, turns into (t, s // t) with strides (d, t * d)
#
def zipped_divide(layout, tiler):
    """cuts every mode of a (flat) layout into tiles, like cute's zipped_divide for simple integer tilers.

    a mode with size s and stride d, tiled by t, turns into (t, s//t) with strides (d, t*d).
    you get ((tile modes...), (rest modes...)), so "where am i inside a tile" and "which tile" """
    tiles, rests, tile_d, rest_d = [], [], [], []
    for s, d, t in zip(layout.shape, layout.stride, tiler):
        if s % t:
            raise ValueError(f"tile {t} doesnt divide a mode of size {s}")
        tiles.append(t); rests.append(s // t); tile_d.append(d); rest_d.append(t * d)
    return Layout((tuple(tiles), tuple(rests)), (tuple(tile_d), tuple(rest_d)))

In [ ]:
Z = zipped_divide(Layout((1, 4, 4)), (1, 2, 2))
print(Z)
print("same windows as pool_layout:", offsets(transpose(Z)) == offsets(pool_layout(1, 4, 4, 2)))

---
## lab 3 - layers = tensors + operators

with layouts, every layer turns into **read thru a layout (gather) + one operator**:

| layer | layout | operator |
|---|---|---|
| convolution | `im2col_layout` | matrix multiply (+ bias) |
| max-pool | `pool_layout` | max of each row |
| fully connected | the weights as rows | matrix multiply (+ bias) |
| flatten | a new shape | nothing at all |

so the one operator that really matters is **matrix multiply**. ours takes two lists of rows and dots every row of A with every row of B: `C[i][j] = A[i] · B[j]`, which is A times B-transposed. cutlass would say both inputs are **K-major**: the numbers being added up (the *K* dimension) sit next to each other in memory. thats why one `sumprod` (a dot product done in c) can do each output.

In [ ]:
# given code - read it, then run it
#
# what this does: cuts a flat list into rows of n, so a flat list can be used as a matrix
# stuff in here:
#   range(0, len(flat_list), n)  0, n, 2n, ... where every row starts
#   flat_list[i:i + n]           the n items starting at i (works on tuples too)
#
def rows(flat_list, n):
    """cuts a flat list into rows of length n, so you can use it as a (len // n) x n matrix"""
    return [flat_list[i:i + n] for i in range(0, len(flat_list), n)]

In [ ]:
# answer (same as solutions/matmul_t.py)
def matmul_t(A, B):
    """C = A times B-transposed for two lists of rows: C[i][j] = dot(A[i], B[j]).

    every row has the same length (the K dimension, the thing being added up). cutlass would say both
    sides are "K-major". gives back C as one flat row-major list of len(A) * len(B) numbers"""
    return [sumprod(a, b) for a in A for b in B]

In [ ]:
checks.matmul_t(matmul_t)

### 3.1 convolution = gather + matmul
lenet's first layer turns into: gather the 784 x 75 patch matrix, then multiply the 6 x 75 filter matrix by it. every image is now a flat list in **CHW** order (all of channel 0, then channel 1, ...), so the output is ready for the next layer.

In [ ]:
# answer (same as solutions/conv2d_forward.py)
def conv2d_forward(x, weight, bias, C, H, W, K):
    """K x K convolution of a C x H x W image (flat CHW list) with Cout filters (flat Cout*C*K*K list).

    the whole layer is: read the patches thru im2col_layout, then one matrix multiply.
    gives back the Cout x OH x OW feature maps as one flat list (CHW order, ready for the next layer)"""
    n = C * K * K                                           # numbers in one patch = numbers in one filter
    patches = rows(gather(x, im2col_layout(C, H, W, K)), n)  # OH*OW rows, one patch per output pixel
    filters = rows(weight, n)                                # Cout rows, one filter each
    out = matmul_t(filters, patches)                         # Cout x (OH*OW), row k = feature map k
    P = len(patches)
    return [v + b for b, fmap in zip(bias, rows(out, P)) for v in fmap]

In [ ]:
checks.conv2d_forward(conv2d_forward)

In [ ]:
x = [v / 255 for v in pixels]                          # the trollface as a flat list, 0..1
t = time.perf_counter(); fast = conv2d_forward(x, params["conv1.w"], params["conv1.b"], 3, 32, 32, 5); t_fast = time.perf_counter() - t
t = time.perf_counter(); slow = conv2d_naive(to_nested(pixels), nest(params["conv1.w"], (6, 3, 5, 5)), params["conv1.b"]); t_slow = time.perf_counter() - t
print("biggest difference from conv2d_naive:", max(abs(a - b) for a, b in zip(fast, flatten_naive(slow))))
print(f"conv2d_naive {t_slow * 1000:.1f} ms   vs   layout + matmul {t_fast * 1000:.1f} ms   ({t_slow / t_fast:.0f}x faster)")

In [ ]:
# answer (same as solutions/maxpool2d_forward.py)
def maxpool2d_forward(x, C, H, W, P=2):
    """P x P max-pooling: read every window thru pool_layout and keep the biggest value of each"""
    windows = gather(x, pool_layout(C, H, W, P))  # window after window, P*P values each
    return [max(w) for w in rows(windows, P * P)]

In [ ]:
checks.maxpool2d_forward(maxpool2d_forward)

In [ ]:
# answer (same as solutions/linear_forward.py)
def linear_forward(x, weight, bias):
    """fully connected layer: out = W x + b. weight is a flat n_out * n_in list (row o = output o's weights)"""
    out = matmul_t(rows(weight, len(x)), [x])  # n_out x 1
    return [v + b for v, b in zip(out, bias)]

In [ ]:
checks.linear_forward(linear_forward)

In [ ]:
# answer (same as solutions/relu_forward.py)
def relu_forward(x):
    """max(0, v) for every number in a flat list"""
    return [v if v > 0.0 else 0.0 for v in x]

In [ ]:
checks.relu_forward(relu_forward)

### 3.2 the fast lenet-5
same network, same weights, same answer, built from your new kernels. notice theres no flatten line: a `16 x 5 x 5` list in CHW order already *is* the 400 number list, so flatten costs nothing.

In [ ]:
# given code - read it, then run it
#
# what this does: the same lenet-5 forward pass as lab 1, built out of your fast kernels on flat lists
# stuff in here:
#   params["conv1.w"]            a flat list now, not nested
#   the 3, 32, 32, 5             channels, height, width, filter size for that conv
#   no flatten line              a 16 x 5 x 5 list in CHW order already IS the 400 long list
#
def lenet_forward(pixels, params):
    """the same lenet-5 forward pass as lenet_forward_naive, on flat lists. gives back 10 class probabilities"""
    x = [v / 255 for v in pixels]
    x = maxpool2d_forward(relu_forward(conv2d_forward(x, params["conv1.w"], params["conv1.b"], 3, 32, 32, 5)), 6, 28, 28)
    x = maxpool2d_forward(relu_forward(conv2d_forward(x, params["conv2.w"], params["conv2.b"], 6, 14, 14, 5)), 16, 10, 10)
    x = relu_forward(linear_forward(x, params["fc1.w"], params["fc1.b"]))   # 400 -> 120 (flatten is free!)
    x = relu_forward(linear_forward(x, params["fc2.w"], params["fc2.b"]))   # 120 -> 84
    return softmax(linear_forward(x, params["fc3.w"], params["fc3.b"]))     # 84 -> 10

In [ ]:
t = time.perf_counter(); fast_probs = lenet_forward(pixels, params); fast_ms = (time.perf_counter() - t) * 1000
print("same answer as lenet_forward_naive:", max(abs(a - b) for a, b in zip(fast_probs, probs)) < 1e-9)
print(f"naive {naive_ms:.0f} ms  ->  layouts + matmul {fast_ms:.1f} ms per image")
macs = 6 * 28 * 28 * 75 + 16 * 10 * 10 * 150 + 400 * 120 + 120 * 84 + 84 * 10   # multiply-adds per image
print(f"lenet-5 does {macs:,} multiply-adds per image: {macs / fast_ms / 1000:.0f} million per second in plain python")

talk about it: a laptop gpu does trillions of multiply-adds a second. why is python so far behind? (every number is a python object and every step goes thru the interpreter. the c code inside `sumprod` and `itemgetter` is the reason lab 3 is ~10x faster than lab 1)

---
## lab 4 - learning: loss, optimizers, learning rate schedules, data loading

the network spits out 10 scores. to learn from that it needs four things:

1. **how wrong was it?** a *loss*: one number, small when the right class gets a high probability
2. **which way should each knob turn?** the *gradient*. lab 5 works that out, here we use toy problems where we already know it
3. **how far should it turn?** the *learning rate* and the *optimizer*
4. **what does it practice on, and in what order?** the *data loader*

### 4.1 cross-entropy loss
$\text{loss} = -\log(p_\text{correct})$. if the network gives the right class 100%, the loss is 0. an even guess over 10 classes costs $\ln 10 \approx 2.30$. being confident and wrong costs alot.

In [ ]:
# answer (same as solutions/cross_entropy_loss.py)
def cross_entropy_loss(logits, label):
    """softmax + cross-entropy: loss = -log(probability of the right class). gives back (loss, probs).

    its written as log(sum(exp(z - m))) + m - z[label] so it never ends up doing log(0)"""
    m = max(logits)
    exps = [math.exp(z - m) for z in logits]
    total = sum(exps)
    loss = math.log(total) + m - logits[label]
    return loss, [e / total for e in exps]

In [ ]:
checks.cross_entropy_loss(cross_entropy_loss)

In [ ]:
for p in (0.99, 0.5, 0.1, 0.01):
    print(f"probability of the right class {p:>4}:  loss {-math.log(p):.2f}")
loss, _ = cross_entropy_loss([math.log(p) for p in probs], label)    # log(probs) works as scores, softmax undoes it
print(f"our untrained network on the {CLASSES[label]}: loss {loss:.2f} (ln 10 = {math.log(10):.2f})")

### 4.2 gradient descent
the gradient says which way is *uphill* for the loss, so step the other way: $w \leftarrow w - \text{lr}\cdot\frac{\partial \text{loss}}{\partial w}$. doing it with the gradient of one example (or a few) at a time is **stochastic gradient descent (sgd)**. its the optimizer in cudagrad and wordhunt-ai, and in this lab.

first a toy problem: fit a line $y = w\,x + b$ thru 5 points by making the mean squared error small. its gradients are easy to work out by hand, theyre in `loss_and_grads`.

In [ ]:
# given code - read it, then run it
#
# what this does: the simplest thing an optimizer can work on, a list of numbers plus their gradient
# stuff in here:
#   list(data)                   copies the numbers into a new list
#   self.grad = None             no gradient yet
#   __repr__                     rounds the numbers when you print it
#
class Param:
    """something to optimize: a flat list of numbers (data) and their gradient (grad, same length)"""

    def __init__(self, data):
        self.data = list(data)
        self.grad = None

    def __repr__(self):
        return f"Param({[round(v, 4) for v in self.data]})"

In [ ]:
xs = [0.0, 1.0, 2.0, 3.0, 4.0]
ys = [1.1, 2.9, 5.2, 7.1, 8.8]                      # roughly y = 2x + 1

def loss_and_grads(w, b):
    errors = [w * x + b - y for x, y in zip(xs, ys)]
    loss = sum(e * e for e in errors) / len(xs)                 # mean squared error
    dw = sum(2 * e * x for e, x in zip(errors, xs)) / len(xs)   # d loss / d w
    db = sum(2 * e for e in errors) / len(xs)                   # d loss / d b
    return loss, dw, db

print(loss_and_grads(0.0, 0.0))   # starting at w = b = 0: big loss, and both gradients say "make them bigger"

In [ ]:
# given code - read it, then run it
#
# what this does: two learning rate schedules, with the same inputs as cudagrad's schedulers.hpp: (t, T, lr_max, lr_min)
# stuff in here:
#   t = how many steps so far, T = how many steps in total
#   min(t / T, 1.0)              how far thru training you are, never more than 1
#
# learning rate schedules, same inputs as cudagrad's schedulers.hpp: (t, T, lr_max, lr_min) -> lr
def constant_lr(t, T, lr_max, lr_min):
    return lr_max


def linear_lr(t, T, lr_max, lr_min):
    return lr_max - (lr_max - lr_min) * min(t / T, 1.0)

In [ ]:
# answer (same as solutions/sgd.py)
class SGD:
    """stochastic gradient descent, like cudagrad's SGD<T>:   w  <-  w - lr * grad

    the learning rate follows a schedule from lr_max (step 0) to lr_min (step total_steps).
    momentum > 0 keeps a running "velocity" v <- momentum * v + grad and steps along v instead"""

    def __init__(self, params, lr_max=0.005, lr_min=1e-5, total_steps=1, schedule=constant_lr, momentum=0.0):
        self.params = list(params)
        self.lr_max, self.lr_min, self.total_steps = lr_max, lr_min, total_steps
        self.schedule, self.momentum = schedule, momentum
        self.velocity = [None] * len(self.params)
        self.t = 0
        self.lr = schedule(0, total_steps, lr_max, lr_min)

    def zero_grad(self):
        for p in self.params:
            p.grad = None

    def step(self):
        lr, mu = self.lr, self.momentum
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            if mu:
                v = self.velocity[i] or [0.0] * len(p.data)
                v = [mu * vi + g for vi, g in zip(v, p.grad)]
                self.velocity[i] = v
                p.data = [w - lr * vi for w, vi in zip(p.data, v)]
            else:
                p.data = [w - lr * g for w, g in zip(p.data, p.grad)]
        self.t += 1
        self.lr = self.schedule(self.t, self.total_steps, self.lr_max, self.lr_min)

In [ ]:
checks.sgd(SGD, Param)

In [ ]:
def fit(lr, steps=60, schedule=constant_lr, momentum=0.0):
    w, b = Param([0.0]), Param([0.0])
    opt = SGD([w, b], lr_max=lr, lr_min=lr / 100, total_steps=steps, schedule=schedule, momentum=momentum)
    losses = []
    for _ in range(steps):
        loss, dw, db = loss_and_grads(w.data[0], b.data[0])
        losses.append(loss)
        w.grad, b.grad = [dw], [db]                    # hand the optimizer the gradients
        opt.step()
    return losses, w.data[0], b.data[0]

runs = {f"lr {lr}": fit(lr)[0] for lr in (0.01, 0.05, 0.14, 0.16)}
viz.line_chart(runs, "sgd fitting a line: the learning rate matters", "step", "loss (log scale)", logy=True)
losses, w, b = fit(0.05)
print(f"lr 0.05 found y = {w:.2f}x + {b:.2f}")

too small is slow. too big overshoots back and forth, and past a certain point it **blows up** (lr 0.16).

try it: find the biggest learning rate that still works. then try `fit(0.01, momentum=0.9)`. momentum keeps a running "velocity", kind of like a ball rolling downhill.

### 4.3 learning rate schedules: cosine annealing
take big steps early (explore) and small steps at the end (settle down). **cosine annealing** slides the learning rate along half a cosine wave from `lr_max` to `lr_min`:

$$\text{lr}(t) = \text{lr}_\min + \tfrac{1}{2}(\text{lr}_\max - \text{lr}_\min)\left(1 + \cos\frac{\pi t}{T}\right)$$

its the `cosine_annealing_LR` in cudagrad's `schedulers.hpp`, and what wordhunt-ai trains with.

In [ ]:
# answer (same as solutions/cosine_annealing_lr.py)
def cosine_annealing_lr(t, T, lr_max, lr_min):
    """cosine annealing: start at lr_max, slide down half a cosine wave, land on lr_min at step T"""
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * min(t / T, 1.0)))

In [ ]:
checks.cosine_annealing_lr(cosine_annealing_lr)

In [ ]:
T = 100
viz.line_chart({name: [f(t, T, 0.01, 0.001) for t in range(T + 1)]
                for name, f in [("constant", constant_lr), ("linear", linear_lr), ("cosine annealing", cosine_annealing_lr)]},
               "learning rate schedules (cudagrad's schedulers.hpp)", "step t", "learning rate")
viz.line_chart({"constant lr 0.14": fit(0.14)[0], "cosine 0.14 -> 0.0014": fit(0.14, schedule=cosine_annealing_lr)[0]},
               "same start, but cosine annealing calms down at the end", "step", "loss (log scale)", logy=True)

### 4.4 data loading
35,000 images x 3,072 numbers = 107 million numbers. as python floats thats several gigabytes, way too much. so the loader keeps the file as compact **bytes** (107 mb) and turns *one image at a time* into floats, only when its needed.

it also does two more jobs:
- **normalize** every colour channel to average 0 and spread 1: $(v/255 - \text{mean}) / \text{std}$. networks learn alot better when the inputs are centered and about the same size
- **shuffle** every epoch so the network cant learn the order

In [ ]:
# given code - read it, then run it
#
# what this does: works out the average and spread of every colour channel over the whole training file.
# thats where the numbers in MEAN and STD come from
# stuff in here:
#   b"".join(...)                glues lots of bytes objects into one
#   sum(plane) / len(plane)      the average byte, then / 255 to get it in 0..1
#   sumprod(plane, plane)        the sum of every value squared
#   var = average(v^2) - average(v)^2   the variance formula. the std is its square root
#   round(x, 4)                  rounds to 4 decimals
#
# average and spread of each colour channel over the training images (pixel / 255).
# channel_stats(DATA / "train.bin") works them out again
MEAN = (0.5863, 0.5382, 0.4765)
STD = (0.2864, 0.2874, 0.3036)


def channel_stats(path):
    """mean and standard deviation of every colour channel over a whole .bin file, in 0..1 units"""
    raw = Path(path).read_bytes()
    n = len(raw) // RECORD
    means, stds = [], []
    for ch in range(3):
        plane = b"".join(raw[i * RECORD + 1 + ch * 1024:i * RECORD + 1 + (ch + 1) * 1024] for i in range(n))
        mean = sum(plane) / len(plane) / 255
        var = sumprod(plane, plane) / len(plane) / 255 ** 2 - mean * mean
        means.append(round(mean, 4))
        stds.append(round(math.sqrt(var), 4))
    return tuple(means), tuple(stds)

In [ ]:
t = time.perf_counter()
print("mean, std per channel:", channel_stats(DATA / "train.bin"), f"({time.perf_counter() - t:.1f} s for 107 million pixels)")

In [ ]:
# answer (same as solutions/dataloader.py)
class DataLoader:
    """hands out (image, label) pairs from a .bin file, one image at a time.

    image = 3,072 floats in CHW order, normalized per channel:  (pixel / 255 - mean) / std
    shuffle=True goes thru the images in a new random order every epoch.
    per_class=N keeps only the first N images of every class (a smaller, balanced training set)"""

    def __init__(self, path, shuffle=False, per_class=None, seed=0):
        self.raw = Path(path).read_bytes()  # the whole file stays as compact bytes, we decode one image at a time
        labels = self.raw[::RECORD]         # every record starts with its label byte
        index = list(range(len(labels)))
        if per_class:
            count, keep = [0] * len(CLASSES), []
            for i in index:
                if count[labels[i]] < per_class:
                    keep.append(i)
                    count[labels[i]] += 1
            index = keep
        self.index = index
        self.labels = [labels[i] for i in index]
        self.shuffle = shuffle
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.index)

    def image(self, k):
        """the k-th image as 3,072 normalized floats"""
        start = self.index[k] * RECORD + 1
        pixels = self.raw[start:start + 3072]
        x = []
        for ch in range(3):
            m, s = MEAN[ch], STD[ch]
            x += [(v / 255 - m) / s for v in pixels[ch * 1024:(ch + 1) * 1024]]
        return x

    def __iter__(self):
        order = list(range(len(self)))
        if self.shuffle:
            self.rng.shuffle(order)
        for k in order:
            yield self.image(k), self.labels[k]

In [ ]:
checks.dataloader(DataLoader)

In [ ]:
train_loader = DataLoader(DATA / "train.bin", shuffle=True, per_class=350)
print(len(train_loader), "training images (350 per class)")

def denormalize(x):   # back to 0..1 so we can look at it
    return [x[c * 1024 + k] * STD[c] + MEAN[c] for c in range(3) for k in range(1024)]

batch = [pair for _, pair in zip(range(16), train_loader)]       # zip with range(16) stops after 16 pairs
viz.show_images([(denormalize(x), CLASSES[y]) for x, y in batch], scale=3, title="synthetic training memes, shuffled")

---
## lab 5 - forward and back propagation

to train, we need to know for **every one of the 62,006 knobs**: if i nudge this knob up a tiny bit, does the loss go up or down, and how fast? thats the gradient $\partial\,\text{loss} / \partial w$. there are two ways to get it:

1. **nudge every knob and re-run the network.** thats 62,006 extra forward passes per image. hopeless for training, but great for *checking*
2. **back propagation.** run forward once, then go *backwards* thru the layers using the **chain rule**. all 62,006 gradients cost about two more forward passes. every deep learning library does this

### 5.1 the chain rule on one neuron

In [ ]:
x, w, b, target = 2.0, 0.5, -0.2, 1.5
z = w * x + b                # forward, one step at a time
y = max(0.0, z)
L = (y - target) ** 2

dL_dy = 2 * (y - target)     # backward: the slope of each step...
dy_dz = 1.0 if z > 0 else 0.0
dz_dw = x
dL_dw = dL_dy * dy_dz * dz_dw   # ...all multiplied together, thats the chain rule

eps = 1e-6                    # check it the slow way: nudge w a tiny bit and watch L
L_nudged = (max(0.0, (w + eps) * x + b) - target) ** 2
print(f"chain rule: {dL_dw:.6f}    nudging: {(L_nudged - L) / eps:.6f}")

### 5.2 the computation graph
a library cant know your formula ahead of time, so it **records** it while it runs forward. every operation makes a new `Tensor` that remembers:
- its **children**: the tensors it was made from
- a **`backward_fn`**: given this tensor's gradient, add the right gradients into its children

thats cudagrad's `Node<T>` (`value`, `children`, `backward_fn`, `op`). then `loss.backward()`:
1. sorts the graph so every tensor comes after the tensors it was made from (a **topological sort**)
2. sets $\partial\,\text{loss}/\partial\,\text{loss} = 1$
3. walks that order **backwards** and calls every `backward_fn`

```cpp
// cudagrad, Node.cuh
void backward() {
    build_topo(this, topo, seen);
    launch_fill<T>(value.grad, T(1), value.size);
    for (auto it = topo.rbegin(); it != topo.rend(); ++it)
        if ((*it)->backward_fn) (*it)->backward_fn();
}
```

why the sort? a tensor thats used twice (say by `b` *and* by `y`) has to collect gradient from *both* uses before it passes anything on. thats also why `add_grad` adds instead of overwriting.

In [ ]:
# given code - read it, then run it
#
# what this does: the Tensor class, one node in the computation graph (cudagrad calls it Node<T>)
# stuff in here:
#   data, shape                  the numbers (flat) and their shape
#   grad                         d(loss)/d(each number), backward() fills it in. starts as None
#   children                     the tensors this one was made from, thats what makes it a graph
#   backward_fn                  a function that sends this tensor's grad back to its children
#   any(c.requires_grad for c in children)   true if at least one child needs a gradient
#   add_grad(g)                  adds g on top of the gradient thats already there (a tensor used twice gets both)
#   @property                    a decorator (a line starting with @ right above a def changes how it works).
#                                this one lets you write t.layout without the () even tho layout is a method
#   leaf(data, shape)            a tensor we want gradients for, like a weight
#
class Tensor:
    """a node in the computation graph: a flat list of numbers + a shape, plus what backprop needs"""

    def __init__(self, data, shape, children=(), op="leaf", requires_grad=False):
        self.data = data                  # the numbers, flat, row-major
        self.shape = tuple(shape)
        self.grad = None                  # d(loss)/d(data), same length as data, backward() fills it in
        self.children = children          # the tensors this one was made from
        self.op = op                      # name of the operation that made it (for printing)
        self.requires_grad = requires_grad or any(c.requires_grad for c in children)
        self.backward_fn = None           # sends self.grad back to the children

    def add_grad(self, g):
        """adds g to this tensor's gradient (a tensor that got used twice collects gradient from both uses)"""
        self.grad = g if self.grad is None else [a + b for a, b in zip(self.grad, g)]

    @property
    def layout(self):
        return Layout(self.shape)

    def __repr__(self):
        head = ", ".join(f"{v:.4f}" for v in self.data[:4]) + (", ..." if len(self.data) > 4 else "")
        return f"Tensor(shape={self.shape}, op={self.op}, data=[{head}])"


def leaf(data, shape):
    """a tensor we want gradients for (a weight), like cudagrad's leaf()"""
    return Tensor(list(data), shape, requires_grad=True)

In [ ]:
# answer (same as solutions/backward.py)
def backward(self):
    """backpropagation: fills in .grad of every tensor this one was made from.

    1. topological order: every tensor comes after the tensors it was made from
    2. d(self)/d(self) = 1
    3. go thru that order backwards and call each tensor's backward_fn (the chain rule, one op at a time)"""
    topo, seen = [], set()

    def build(node):
        if id(node) not in seen:
            seen.add(id(node))
            for child in node.children:
                build(child)
            topo.append(node)
    build(self)
    self.grad = [1.0] * len(self.data)
    for node in reversed(topo):
        if node.backward_fn is not None:
            node.backward_fn()


Tensor.backward = backward

In [ ]:
checks.backward(Tensor)

### 5.3 the backward kernels
for every layer: given `dy` (the gradient of the loss for the layer's output), work out the gradients for its input and its weights. like cudagrad's `launch_*_backward` kernels, these are plain functions on flat lists.

**relu.** $y = \max(0, x)$ has slope 1 where $x > 0$ and 0 everywhere else, so the gradient either goes straight thru or gets blocked.

In [ ]:
# answer (same as solutions/relu_backward.py)
def relu_backward(dy, x):
    """y = relu(x). the gradient goes thru where x > 0 and gets blocked (0) where x <= 0"""
    return [g if v > 0.0 else 0.0 for g, v in zip(dy, x)]

In [ ]:
checks.relu_backward(relu_backward)

**fully connected.** $y = Wx + b$, so $y_o = b_o + \sum_i W_{oi}\,x_i$. every $x_i$ touches every $y_o$ with weight $W_{oi}$, so:

$$\frac{\partial L}{\partial x_i} = \sum_o dy_o\,W_{oi}\;(\text{thats } W^T dy),\qquad \frac{\partial L}{\partial W_{oi}} = dy_o\,x_i,\qquad \frac{\partial L}{\partial b_o} = dy_o$$

$W^T$ is just W's memory read thru the transposed **layout**, nothing gets copied untill we gather it.

In [ ]:
# answer (same as solutions/linear_backward.py)
def linear_backward(dy, x, weight):
    """y = W x + b with W stored as n_out rows of n_in. given dy = d(loss)/dy, gives back (dx, dweight, dbias).

    dx = W^T dy           (column i of W dotted with dy)
    dW = dy x^T           (every dy times every x: dW[o][i] = dy[o] * x[i])
    db = dy"""
    n_in, n_out = len(x), len(dy)
    columns = rows(gather(weight, Layout((n_in, n_out), (1, n_in))), n_out)  # the transpose of W, as rows
    dx = matmul_t(columns, [dy])
    dweight = [g * v for g in dy for v in x]
    return dx, dweight, list(dy)

In [ ]:
checks.linear_backward(linear_backward)

**max-pool.** only the winner of each window had any effect on the output, so only the winner gets the gradient. `pool_layout` tells you where the winner lives in memory.

In [ ]:
# answer (same as solutions/maxpool2d_backward.py)
def maxpool2d_backward(dy, x, C, H, W, P=2):
    """y = maxpool(x). each window's gradient goes to the one pixel that won the max, the rest get 0"""
    L = pool_layout(C, H, W, P)
    windows = rows(gather(x, L), P * P)
    offs = offsets(L)
    dx = [0.0] * len(x)
    for k, (g, w) in enumerate(zip(dy, windows)):
        winner = w.index(max(w))           # which of the P*P spots had the max
        dx[offs[k * P * P + winner]] += g  # where that spot lives in x
    return dx

In [ ]:
checks.maxpool2d_backward(maxpool2d_backward)

**convolution.** the forward pass was `out = filters @ patches^T` (+ bias), a matrix multiply, so its backward pass follows the same rules as the fully connected layer:

| gradient of | is | shape |
|---|---|---|
| bias | the sum of each feature map's `dy` | Cout |
| filters | `dy @ patches` | Cout x 75 |
| patches | `dy^T @ filters` | 784 x 75 |
| image | every patch gradient **added back** onto the pixel it was read from | 3 x 32 x 32 |

written out with indices:

$$db[k] = \sum_{i,j} dy[k][i][j] \qquad dw[k][c][u][v] = \sum_{i,j} dy[k][i][j]\; x[c][i+u][j+v] \qquad dx[c][h][w] = \sum_k \sum_{u,v} dy[k][h-u][w-v]\; w[k][c][u][v]$$

that last row of the table is the deep idea: **the backward of a gather is a scatter-add thru the same layout.** a pixel that showed up in 25 patches collects gradient from all 25.

(the first layer doesnt need the image's gradient, the image isnt a knob. so `need_dx=False` skips that work)

In [ ]:
# answer (same as solutions/conv2d_backward.py)
def conv2d_backward(dy, x, weight, C, H, W, K, need_dx=True):
    """y = conv2d_forward(x, weight, bias). given dy (Cout x OH x OW), gives back (dx, dweight, dbias).

    forward was  out = filters @ patches^T  (+ a bias per filter), so by the matmul rules:
      dbias[k]    = the sum of feature map k's gradient
      dweight     = dy @ patches              (Cout x n, every weight saw one pixel per output spot)
      dpatches    = dy^T @ filters            (P x n, how much each patch entry mattered)
      dx          = add every dpatches entry back onto the pixel it was read from (the same layout!)"""
    n = C * K * K
    L = im2col_layout(C, H, W, K)
    P = size(L[0])                                 # output pixels
    Cout = len(weight) // n
    dy_rows = rows(dy, P)                          # Cout rows, one feature map's gradient each
    dbias = [sum(r) for r in dy_rows]
    patch_columns = rows(gather(x, transpose(L)), P)    # n rows: patch entry j at every output spot
    dweight = matmul_t(dy_rows, patch_columns)          # Cout x n
    if not need_dx:                                # the first layer's input is the image, nobody needs its gradient
        return None, dweight, dbias
    dy_columns = rows(gather(dy, Layout((P, Cout), (1, P))), Cout)       # P rows: every filter's dy at spot p
    filter_columns = rows(gather(weight, Layout((n, Cout), (1, n))), Cout)  # n rows: entry j of every filter
    dpatches = matmul_t(dy_columns, filter_columns)  # P x n, same order as offsets(L)
    dx = [0.0] * len(x)
    for o, g in zip(offsets(L), dpatches):         # scatter-add, the backward of a gather
        dx[o] += g
    return dx, dweight, dbias

In [ ]:
checks.conv2d_backward(conv2d_backward)

**softmax + cross-entropy.** the calculus works out to something realy simple: $\partial L / \partial z_k = p_k - [k = \text{label}]$. the network's probabilities, minus 1 at the right answer.

In [ ]:
# answer (same as solutions/cross_entropy_backward.py)
def cross_entropy_backward(probs, label):
    """d(loss)/d(logits) for softmax + cross-entropy: just probs - one_hot(label)"""
    d = list(probs)
    d[label] -= 1.0
    return d

In [ ]:
checks.cross_entropy_backward(cross_entropy_backward)

### 5.4 wiring it together
every operation runs its forward kernel, wraps the answer in a `Tensor` that remembers its children, and sticks on a `backward_fn` that calls the matching backward kernel. same pattern as cudagrad's `Node.cuh`.

In [ ]:
# given code - read it, then run it
#
# what this does: wires your forward and backward kernels into the graph. every op does 3 things:
#   1. runs the forward kernel and wraps the answer in a new Tensor that remembers its children
#   2. defines backward_fn, a function INSIDE the op. even after the op returns, backward_fn can still see
#      x, weight, out and so on (thats called a closure), so later it can run the backward kernel
#      with the right inputs
#   3. sticks backward_fn onto the output tensor
# stuff in here:
#   C, H, W = x.shape            unpacking the shape
#   out.grad                     the gradient that shows up at the output when backward() runs
#   x.requires_grad              skips work for stuff that doesnt need gradients (like the input picture)
#   flatten                      makes a new Tensor with the SAME data list, so nothing gets copied
#   out.probs = probs            you can stick any extra attribute onto an object
#
# the autograd operations. each one runs a forward kernel, wraps the answer in a Tensor that remembers its
# children, and sticks on a backward_fn that runs the matching backward kernel (cudagrad's Node.cuh pattern)
def relu(x):
    out = Tensor(relu_forward(x.data), x.shape, (x,), "relu")

    def backward_fn():
        x.add_grad(relu_backward(out.grad, x.data))
    out.backward_fn = backward_fn
    return out


def linear(x, weight, bias):
    """cudagrad: bias_add(matmul(x, W), b)"""
    out = Tensor(linear_forward(x.data, weight.data, bias.data), bias.shape, (x, weight, bias), "linear")

    def backward_fn():
        dx, dw, db = linear_backward(out.grad, x.data, weight.data)
        if x.requires_grad:
            x.add_grad(dx)
        weight.add_grad(dw)
        bias.add_grad(db)
    out.backward_fn = backward_fn
    return out


def conv2d(x, weight, bias):
    """cudagrad: conv_bias(conv(x, filters), bias)"""
    C, H, W = x.shape
    Cout, _, K, _ = weight.shape
    out = Tensor(conv2d_forward(x.data, weight.data, bias.data, C, H, W, K),
                 (Cout, H - K + 1, W - K + 1), (x, weight, bias), "conv2d")

    def backward_fn():
        dx, dw, db = conv2d_backward(out.grad, x.data, weight.data, C, H, W, K, need_dx=x.requires_grad)
        if x.requires_grad:
            x.add_grad(dx)
        weight.add_grad(dw)
        bias.add_grad(db)
    out.backward_fn = backward_fn
    return out


def maxpool2d(x, p=2):
    C, H, W = x.shape
    out = Tensor(maxpool2d_forward(x.data, C, H, W, p), (C, H // p, W // p), (x,), "maxpool2d")

    def backward_fn():
        x.add_grad(maxpool2d_backward(out.grad, x.data, C, H, W, p))
    out.backward_fn = backward_fn
    return out


def flatten(x):
    """a view: the same numbers with a new shape. nothing gets copied and the gradient goes straight thru"""
    out = Tensor(x.data, (len(x.data),), (x,), "flatten")

    def backward_fn():
        x.add_grad(out.grad)
    out.backward_fn = backward_fn
    return out


def softmax_cross_entropy(logits, label):
    """the loss as a tensor with one number in it (cudagrad's softmax_cross_entropy)"""
    loss, probs = cross_entropy_loss(logits.data, label)
    out = Tensor([loss], (1,), (logits,), "softmax_cross_entropy")
    out.probs = probs

    def backward_fn():
        logits.add_grad([g * out.grad[0] for g in cross_entropy_backward(probs, label)])
    out.backward_fn = backward_fn
    return out

In [ ]:
# a gradient check: backprop vs "nudge and measure" for one conv -> relu -> max-pool on a random 2x6x6 input
random.seed(1)
xin = leaf([random.uniform(-1, 1) for _ in range(2 * 6 * 6)], (2, 6, 6))
wc = leaf([random.uniform(-1, 1) for _ in range(3 * 2 * 3 * 3)], (3, 2, 3, 3))
bc = leaf([0.1, 0.0, -0.1], (3,))

def forward():
    return maxpool2d(relu(conv2d(xin, wc, bc)))   # the "loss" here is just the sum of this output

out = forward()
out.backward()                                    # backward() starts from all ones: d(sum)/d(out) = 1
eps = 1e-6
for name, t in [("input", xin), ("weight", wc), ("bias", bc)]:
    i = random.choice([k for k, g in enumerate(t.grad) if g != 0.0])     # pick an entry that matters
    old = t.data[i]
    t.data[i] = old + eps; up = sum(forward().data)
    t.data[i] = old - eps; down = sum(forward().data)
    t.data[i] = old
    print(f"{name:>6}[{i:>2}]   backprop {t.grad[i]: .6f}    nudging {(up - down) / (2 * eps): .6f}")

---
## lab 6 - train lenet-5 on memes

### 6.1 modules
layers that own their weights, stacked in a container. these copy cudagrad's `Conv`, `MaxPool`, `Flatten`, `Linear` and `CNN` (`net.add(...).add(...)`).

In [ ]:
# given code - read it, then run it
#
# what this does: layers that own their weights (like cudagrad's Conv, MaxPool, Flatten, Linear) and a CNN
# that stacks them into lenet-5
# stuff in here:
#   class Conv(Module)           Conv is a kind of Module, it gets __call__ and parameters() from Module for free
#                                (thats called inheritance)
#   def __call__(self, x)        so model(x) runs model.forward(x)
#   return self                  add() gives back the CNN itself, so you can chain .add(...).add(...)
#   [p for layer in self.layers for p in layer.parameters()]   every weight in the whole network in one list
#   max(range(n), key=probs.__getitem__)    argmax: the position of the biggest probability
#   json.dumps(...), json.loads(...)        turn lists and dicts into text and back, thats how weights get saved
#   float(f"{v:.7g}")            rounds a number to 7 digits so the file is smaller
#   Path(path).parent.mkdir(parents=True, exist_ok=True)   makes the folder if it isnt there yet
#   **info                       collects any extra name=value arguments into a dict
#
class Module:
    """a layer: something you call on a tensor that might own weights (cudagrad's Module<T>)"""

    def __call__(self, x):
        return self.forward(x)

    def parameters(self):
        return []


class Conv(Module):
    """cudagrad's Conv: K x K filters with he init, zero bias, then relu"""

    def __init__(self, c_in, c_out, k, relu=True):
        self.weight = leaf(he_init(c_out * c_in * k * k, c_in * k * k), (c_out, c_in, k, k))
        self.bias = leaf([0.0] * c_out, (c_out,))
        self.use_relu = relu

    def forward(self, x):
        y = conv2d(x, self.weight, self.bias)
        return relu(y) if self.use_relu else y

    def parameters(self):
        return [self.weight, self.bias]


class MaxPool(Module):
    def __init__(self, p=2):
        self.p = p

    def forward(self, x):
        return maxpool2d(x, self.p)


class Flatten(Module):
    def forward(self, x):
        return flatten(x)


class Linear(Module):
    """cudagrad's Linear: fully connected, he init, zero bias, relu if you want it"""

    def __init__(self, n_in, n_out, relu=True):
        self.weight = leaf(he_init(n_out * n_in, n_in), (n_out, n_in))
        self.bias = leaf([0.0] * n_out, (n_out,))
        self.use_relu = relu

    def forward(self, x):
        y = linear(x, self.weight, self.bias)
        return relu(y) if self.use_relu else y

    def parameters(self):
        return [self.weight, self.bias]


class CNN(Module):
    """a stack of layers, like cudagrad's CNN: net.add(layer).add(layer)..."""

    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)
        return self

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def predict(self, image):
        """(guessed label, probabilities) for one normalized image (3,072 floats)"""
        probs = softmax(self(Tensor(image, (3, 32, 32))).data)
        return max(range(len(probs)), key=probs.__getitem__), probs

    def save(self, path, **info):
        params = [{"shape": p.shape, "data": [float(f"{v:.7g}") for v in p.data]} for p in self.parameters()]
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        Path(path).write_text(json.dumps({"model": "lenet5", "classes": CLASSES, "mean": MEAN, "std": STD,
                                          "info": info, "params": params}))

    def load(self, path):
        saved = json.loads(Path(path).read_text())
        for p, s in zip(self.parameters(), saved["params"]):
            if tuple(s["shape"]) != p.shape:
                raise ValueError(f"shape mismatch: the file has {s['shape']}, the model has {p.shape}")
            p.data = list(s["data"])
        return saved.get("info", {})


def lenet5(num_classes=10, seed=0):
    """lenet-5 for 3 x 32 x 32 images (lecun et al. 1998, with relu and max-pooling)"""
    random.seed(seed)
    return (CNN()
            .add(Conv(3, 6, 5)).add(MaxPool(2))      # 3x32x32 -> 6x28x28 -> 6x14x14
            .add(Conv(6, 16, 5)).add(MaxPool(2))     #         -> 16x10x10 -> 16x5x5
            .add(Flatten())                          #         -> 400
            .add(Linear(400, 120))                   #         -> 120
            .add(Linear(120, 84))                    #         -> 84
            .add(Linear(84, num_classes, relu=False)))  #      -> 10 scores ("logits")

In [ ]:
model = lenet5(seed=0)
print(sum(len(p.data) for p in model.parameters()), "numbers to learn in", len(model.parameters()), "tensors")
logits = model(Tensor(DataLoader(DATA / "test.bin").image(90), (3, 32, 32)))
print(logits)

before training, make sure backprop thru the *whole* network matches nudging, on random knobs from every layer:

In [ ]:
checks.gradcheck_lenet(lenet5, Tensor, softmax_cross_entropy, DataLoader)

### 6.2 one step of learning
forward → loss → clear the old gradients → backward → step. those five lines are the core of every neural network ever trained, from lenet to chatgpt.

In [ ]:
# answer (same as solutions/train_step.py)
def train_step(model, opt, image, label):
    """one step of learning on one image. gives back (loss, was the guess right?)"""
    logits = model(Tensor(image, (3, 32, 32)))      # 1. forward propagation
    loss = softmax_cross_entropy(logits, label)     # 2. how wrong were we
    opt.zero_grad()                                 # 3. forget the last step's gradients
    loss.backward()                                 # 4. back propagation: d(loss)/d(every weight)
    opt.step()                                      # 5. nudge every weight downhill
    guess = max(range(len(logits.data)), key=logits.data.__getitem__)
    return loss.data[0], guess == label

In [ ]:
checks.train_step(train_step, lenet5, SGD, DataLoader)

In [ ]:
test_loader = DataLoader(DATA / "test.bin")
image, true = test_loader.image(105), test_loader.labels[105]      # test image 105 is a surprised pikachu
model = lenet5(seed=0)
opt = SGD(model.parameters(), lr_max=0.005)
before = model.predict(image)[1][true]
steps = [train_step(model, opt, image, true) for _ in range(15)]
viz.line_chart({"loss": [s[0] for s in steps]}, "15 steps on ONE image: the network memorizes it", "step", "loss")
print(f"P({CLASSES[true]}) went from {before:.0%} to {model.predict(image)[1][true]:.0%}")

### 6.3 measuring accuracy
**accuracy** = the fraction of images where the highest score is the right class. a **confusion matrix** shows *which* mistakes it makes: row = the right answer, column = the guess.

In [ ]:
# answer (same as solutions/evaluate.py)
def evaluate(model, loader, limit=None):
    """accuracy on a dataset, plus a confusion matrix: confusion[true label][guessed label] = count"""
    confusion = [[0] * len(CLASSES) for _ in CLASSES]
    for k, (image, label) in enumerate(loader):
        if limit is not None and k >= limit:
            break
        guess, _ = model.predict(image)
        confusion[label][guess] += 1
    total = sum(map(sum, confusion))
    correct = sum(confusion[i][i] for i in range(len(CLASSES)))
    return correct / max(total, 1), confusion

In [ ]:
checks.evaluate(evaluate, lenet5, DataLoader)

### 6.4 train it
one image at a time, plain sgd, cosine annealed learning rate. thats the same recipe as wordhunt-ai. plain python does about 45-60 images a second on a fast laptop (an older one maybe a third of that), so pick a size that fits your time:

| `PER_CLASS` | `EPOCHS` | `LR` | images seen | time: fast laptop (measured) → older laptop (guess) | real meme accuracy |
|---|---|---|---|---|---|
| 100 | 5 | 0.01 | 5,000 | ~2 min → ~6 min | ~77% |
| 350 | 4 | 0.005 | 14,000 | ~4-6 min → ~15 min | ~83% |
| `None` (all) | 4 | 0.005 | 140,000 | ~46 min → 2-3 hours | ~89% (see 6.6) |

while it trains: talk about what the network might be looking at, or peek at the bonus notebooks.

In [ ]:
# given code - read it, then run it
#
# what this does: the training loop. sgd with a cosine learning rate, one image per step, and it prints progress
# stuff in here:
#   schedule or cosine_annealing_lr   "or" picks the second one when the first one is None
#   time.time()                  seconds since 1970. take it twice and subtract to time something
#   history.append({...})        a list of dicts, one per printed line, used for the charts later
#   f"{x:.3f}", f"{x:6.1%}"      3 decimals, and a percent 6 characters wide with 1 decimal
#   print(..., flush=True)       shows the line right away instead of waiting
#   evaluate(model, val_loader, limit=val_limit)   checks progress on some val images after every epoch
#
def train(model, train_loader, epochs=1, lr_max=0.005, lr_min=1e-5, schedule=None, val_loader=None,
          val_limit=500, log_every=500, history=None):
    """trains with plain sgd and a cosine annealing learning rate, one image at a time (no batching).
    prints progress and gives back a list of {step, lr, loss, acc} records (also added onto `history`)"""
    opt = SGD(model.parameters(), lr_max, lr_min, total_steps=epochs * len(train_loader),
              schedule=schedule or cosine_annealing_lr)
    history = [] if history is None else history
    start = time.time()
    for epoch in range(1, epochs + 1):
        run_loss, run_right, seen = 0.0, 0, 0
        for image, label in train_loader:
            lr = opt.lr
            loss, right = train_step(model, opt, image, label)
            run_loss += loss
            run_right += right
            seen += 1
            if seen % log_every == 0 or seen == len(train_loader):
                rate = opt.t / (time.time() - start)
                history.append({"step": opt.t, "lr": lr, "loss": run_loss / seen, "acc": run_right / seen})
                print(f"epoch {epoch}/{epochs}  image {seen:>6}/{len(train_loader)}  loss {run_loss / seen:.3f}  "
                      f"train acc {run_right / seen:6.1%}  lr {lr:.5f}  ({rate:.0f} images/s)", flush=True)
        if val_loader is not None:
            acc, _ = evaluate(model, val_loader, limit=val_limit)
            print(f"  -> epoch {epoch}: validation accuracy {acc:.1%} (first {val_limit} val images)", flush=True)
    print(f"done: {opt.t} steps in {time.time() - start:.0f} s")
    return history

In [ ]:
PER_CLASS = 350      # images per class. 100 = quick, 350 = the class run, None = everything
EPOCHS = 4
LR = 0.005           # lr_max, cosine annealing takes it down to 1e-5

model = lenet5(seed=0)
train_loader = DataLoader(DATA / "train.bin", shuffle=True, per_class=PER_CLASS)
val_loader = DataLoader(DATA / "val.bin")
print(f"{len(train_loader):,} images x {EPOCHS} epochs = {len(train_loader) * EPOCHS:,} steps of sgd")
history = train(model, train_loader, EPOCHS, lr_max=LR, val_loader=val_loader, val_limit=500)

In [ ]:
viz.line_chart({"training loss (running average inside each epoch)": [(h["step"], h["loss"]) for h in history]},
               "learning curve", "sgd step", "loss")
viz.line_chart({"learning rate": [(h["step"], h["lr"]) for h in history]}, "cosine annealing", "sgd step", "lr")

### 6.5 the real exam: 150 real memes
the network only ever practiced on *synthetic* memes. does it work on real ones from the internet?

In [ ]:
acc, confusion = evaluate(model, test_loader)
print(f"accuracy on 150 real memes it has never seen: {acc:.1%}   (random guessing: 10%)")
viz.show_confusion(confusion, CLASSES)

In [ ]:
import csv
originals = [row["original"] for row in csv.DictReader(open(DATA / "test.csv"))]   # full size file for every test image

def show(*indices, net=None):
    net = net or model
    viz.show_predictions([(DATA / originals[i], test[i][1], net.predict(test_loader.image(i))[1],
                           test_loader.labels[i], CLASSES) for i in indices])

show(12, 16, 32, 46, 61, 77, 91, 106, 121, 136)    # one per class (12 = six_seven: someone doing the hands)

try it: `show(7)`, `show(20, 21, 22)`... test images 0-14 are six_seven, 15-29 tung tung tung sahur, 30-44 tralalero tralala, 45-59 skibidi toilet, 60-74 chill guy, 75-89 doge, 90-104 trollface, 105-119 surprised pikachu, 120-134 drake, 135-149 distracted boyfriend. which mistakes make sense? (hint: at 32 x 32 the network leans on **colour** alot. a person in a blue shirt can look like the blue shark)

### 6.6 a fully trained model
your teacher trained the *same code* on all 35,000 images for 4 epochs (`weights/lenet_meme10.json` says how long it took).

In [ ]:
pretrained = lenet5()
info = pretrained.load("weights/lenet_meme10.json")
print(info)
acc_pre, confusion_pre = evaluate(pretrained, test_loader)
print(f"pretrained on all 35,000 images: {acc_pre:.1%} on the real memes")
viz.show_confusion(confusion_pre, CLASSES, "pretrained model (rows = the right answer, columns = the guess)")
show(2, 4, 5, 10, 12, net=pretrained)              # some six_seven memes

In [ ]:
model.save("weights/my_lenet.json", note="trained in class")   # load it again any time with lenet5().load(...)
print("saved weights/my_lenet.json")

---
## thats it, you built a neural network from scratch

everything that just ran is your code (or code you read) plus the python standard library: layouts and gathers, one matrix multiply, a topological sort, the chain rule, sgd with cosine annealing, and a data loader. thats the same skeleton pytorch, cudagrad and infanttorch have. they just run it on gpus.

### talk about it: how would we make it faster?
- **where does the time go?** each training step is ~0.65 million multiply-adds forward and ~1.3 million backward, plus updating all 62,006 weights *after every single image*
- **batching**: average the gradient over N images and update once. one bigger matrix multiply instead of N small ones → [bonus A](addon_A_batching.ipynb)
- **split-k reduction**: a weight's gradient is a giant sum over every image and every position (the *K* dimension). cut K into slices, add up each slice on a different core (or gpu thread block), then add the partial sums together → [bonus B](addon_B_split_k.ipynb)
- **gpus**: thousands of cores. cudagrad trains a similar cnn on 124,800 images in ~6 seconds per epoch on an rtx 4080. cute layouts like the ones you wrote decide which gpu thread reads which number

more reading: [cudagrad](https://github.com/notYnoj/cudagrad) (cuda autograd with split-k kernels) · [infanttorch](https://github.com/TorchLearners/infantTorch) · [wordhunt-ai](https://github.com/notYnoj/WordHunt-AI) (a lenet style cnn in plain c++) · [cute docs](https://github.com/NVIDIA/cutlass/blob/main/media/docs/cpp/cute/01_layout.md)